In [1]:
!git clone https://github.com/ultralytics/yolov5.git


Cloning into 'yolov5'...
remote: Enumerating objects: 17265, done.
remote: Counting objects: 100% (105/105), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 17265 (delta 57), reused 14 (delta 14), pack-reused 17160 (from 4)
Receiving objects: 100% (17265/17265), 16.01 MiB | 21.24 MiB/s, done.
Resolving deltas: 100% (11799/11799), done.


In [2]:
%cd yolov5

/kaggle/working/yolov5


In [3]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 910.1/910.1 kB 14.3 MB/s eta 0:00:00a 0:00:01


In [1]:
import os
import shutil
import pandas as pd
from PIL import Image
import subprocess
import matplotlib.pyplot as plt

In [5]:
BASE_DIR = "/kaggle/working"
DATASET_CSV = "/kaggle/input/car-brand/CarDataset.csv"  # Update with your dataset path
IMAGES_DIR = "/kaggle/input/car-brand/Public/Public"  # Update with your images path
OUTPUT_DIR = os.path.join(BASE_DIR, "car_brand_dataset")

TRAIN_CSV = ['/kaggle/input/car-brand/CarDataset-Splits-1-Train.csv',
             '/kaggle/input/car-brand/CarDataset-Splits-2-Train.csv',
             '/kaggle/input/car-brand/CarDataset-Splits-3-Train.csv',
             '/kaggle/input/car-brand/CarDataset-Splits-4-Train.csv',
             '/kaggle/input/car-brand/CarDataset-Splits-5-Train.csv'
            ]
TEST_CSV = ['/kaggle/input/car-brand/CarDataset-Splits-1-Test.csv',
             '/kaggle/input/car-brand/CarDataset-Splits-2-Test.csv',
             '/kaggle/input/car-brand/CarDataset-Splits-3-Test.csv',
             '/kaggle/input/car-brand/CarDataset-Splits-4-Test.csv',
             '/kaggle/input/car-brand/CarDataset-Splits-5-Test.csv'
            ]

DEVICE = "0, 1"
EPOCHS = 30
BATCH_SIZE = 32

In [6]:
def organize_dataset(train_csv, test_csv, images_dir, output_dir):
    """
    Organizes dataset into train and validation folders for YOLOv5 training.
    Args:
        train_csv: CSV file containing training image paths and labels.
        test_csv: CSV file containing validation image paths and labels.
        images_dir: Directory where images are stored.
        output_dir: Directory to store organized dataset.
    Returns:
        Tuple of paths to train and validation directories.
    """
    # Prepare paths for train and validation directories
    train_dir = os.path.join(output_dir, "train")
    val_dir = os.path.join(output_dir, "val")
    
    # Create train and val folders
    for folder in [train_dir, val_dir]:
        os.makedirs(folder, exist_ok=True)

    # Process training data
    train_data = pd.read_csv(train_csv, header=None, names=['image_path', 'label'])
    train_data['label'] = train_data['label'].astype(str)
    for label, group in train_data.groupby("label"):
        label_train_dir = os.path.join(train_dir, label)
        os.makedirs(label_train_dir, exist_ok=True)
        for _, row in group.iterrows():
            img_path = os.path.join(images_dir, row["image_path"])
            shutil.copy(img_path, label_train_dir)

    # Process validation data
    test_data = pd.read_csv(test_csv, header=None, names=['image_path', 'label'])
    test_data['label'] = test_data['label'].astype(str)
    for label, group in test_data.groupby("label"):
        label_val_dir = os.path.join(val_dir, label)
        os.makedirs(label_val_dir, exist_ok=True)
        for _, row in group.iterrows():
            img_path = os.path.join(images_dir, row["image_path"])
            shutil.copy(img_path, label_val_dir)

    return train_dir, val_dir


In [7]:
%%writefile '/kaggle/working/yolov5/classify/val.py'
# Ultralytics YOLOv5 🚀, AGPL-3.0 license
"""
Validate a trained YOLOv5 classification model on a classification dataset.

Usage:
    $ bash data/scripts/get_imagenet.sh --val  # download ImageNet val split (6.3G, 50000 images)
    $ python classify/val.py --weights yolov5m-cls.pt --data ../datasets/imagenet --img 224  # validate ImageNet

Usage - formats:
    $ python classify/val.py --weights yolov5s-cls.pt                 # PyTorch
                                       yolov5s-cls.torchscript        # TorchScript
                                       yolov5s-cls.onnx               # ONNX Runtime or OpenCV DNN with --dnn
                                       yolov5s-cls_openvino_model     # OpenVINO
                                       yolov5s-cls.engine             # TensorRT
                                       yolov5s-cls.mlmodel            # CoreML (macOS-only)
                                       yolov5s-cls_saved_model        # TensorFlow SavedModel
                                       yolov5s-cls.pb                 # TensorFlow GraphDef
                                       yolov5s-cls.tflite             # TensorFlow Lite
                                       yolov5s-cls_edgetpu.tflite     # TensorFlow Edge TPU
                                       yolov5s-cls_paddle_model       # PaddlePaddle
"""

import argparse
import os
import sys
from pathlib import Path

import torch
from tqdm import tqdm

import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, classification_report

FILE = Path(__file__).resolve()
ROOT = FILE.parents[1]  # YOLOv5 root directory
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))  # add ROOT to PATH
ROOT = Path(os.path.relpath(ROOT, Path.cwd()))  # relative

from models.common import DetectMultiBackend
from utils.dataloaders import create_classification_dataloader
from utils.general import (
    LOGGER,
    TQDM_BAR_FORMAT,
    Profile,
    check_img_size,
    check_requirements,
    colorstr,
    increment_path,
    print_args,
)
from utils.torch_utils import select_device, smart_inference_mode


@smart_inference_mode()
def run(
    data=ROOT / "../datasets/mnist",  # dataset dir
    weights=ROOT / "yolov5s-cls.pt",  # model.pt path(s)
    batch_size=128,  # batch size
    imgsz=224,  # inference size (pixels)
    device="",  # cuda device, i.e. 0 or 0,1,2,3 or cpu
    workers=8,  # max dataloader workers (per RANK in DDP mode)
    verbose=False,  # verbose output
    project=ROOT / "runs/val-cls",  # save to project/name
    name="exp",  # save to project/name
    exist_ok=False,  # existing project/name ok, do not increment
    half=False,  # use FP16 half-precision inference
    dnn=False,  # use OpenCV DNN for ONNX inference
    model=None,
    dataloader=None,
    criterion=None,
    pbar=None,
):
    """Validates a YOLOv5 classification model on a dataset, computing metrics like top1 and top5 accuracy."""
    # Initialize/load model and set device
    training = model is not None
    if training:  # called by train.py
        device, pt, jit, engine = next(model.parameters()).device, True, False, False  # get model device, PyTorch model
        half &= device.type != "cpu"  # half precision only supported on CUDA
        model.half() if half else model.float()
    else:  # called directly
        device = select_device(device, batch_size=batch_size)

        save_dir = None
        if save_dir is None:
            save_dir = increment_path(Path(project) / name, exist_ok=exist_ok)
            save_dir.mkdir(parents=True, exist_ok=True)

        # Load model
        model = DetectMultiBackend(weights, device=device, dnn=dnn, fp16=half)
        stride, pt, jit, engine = model.stride, model.pt, model.jit, model.engine
        imgsz = check_img_size(imgsz, s=stride)  # check image size
        half = model.fp16  # FP16 supported on limited backends with CUDA
        if engine:
            batch_size = model.batch_size
        else:
            device = model.device
            if not (pt or jit):
                batch_size = 1  # export.py models default to batch-size 1
                LOGGER.info(f"Forcing --batch-size 1 square inference (1,3,{imgsz},{imgsz}) for non-PyTorch models")

        # Dataloader
        data = Path(data)
        test_dir = data / "test" if (data / "test").exists() else data / "val"  # data/test or data/val
        dataloader = create_classification_dataloader(
            path=test_dir, imgsz=imgsz, batch_size=batch_size, augment=False, rank=-1, workers=workers
        )

    model.eval()
    y_true, y_pred = [], []  # Collect labels and predictions
    pred, targets, loss, dt = [], [], 0, (Profile(device=device), Profile(device=device), Profile(device=device))
    n = len(dataloader)  # number of batches
    action = "validating" if dataloader.dataset.root.stem == "val" else "testing"
    desc = f"{pbar.desc[:-36]}{action:>36}" if pbar else f"{action}"
    bar = tqdm(dataloader, desc, n, not training, bar_format=TQDM_BAR_FORMAT, position=0)
    with torch.cuda.amp.autocast(enabled=device.type != "cpu"):
        for images, labels in bar:
            with dt[0]:
                images, labels = images.to(device, non_blocking=True), labels.to(device)

            with dt[1]:
                y = model(images)

            with dt[2]:
                pred.append(y.argsort(1, descending=True)[:, :5])
                targets.append(labels)

                # Collect predictions and true labels
                _, preds = torch.max(y, 1)
                y_pred.extend(preds.cpu().numpy())
                y_true.extend(labels.cpu().numpy())

                if criterion:
                    loss += criterion(y, labels)

    loss /= n
    pred, targets = torch.cat(pred), torch.cat(targets)
    correct = (targets[:, None] == pred).float()
    acc = torch.stack((correct[:, 0], correct.max(1).values), dim=1)  # (top1, top5) accuracy
    top1, top5 = acc.mean(0).tolist()

    if pbar:
        pbar.desc = f"{pbar.desc[:-36]}{loss:>12.3g}{top1:>12.3g}{top5:>12.3g}"
    if verbose:  # all classes
        LOGGER.info(f"{'Class':>24}{'Images':>12}{'top1_acc':>12}{'top5_acc':>12}")
        LOGGER.info(f"{'all':>24}{targets.shape[0]:>12}{top1:>12.3g}{top5:>12.3g}")
        for i, c in model.names.items():
            acc_i = acc[targets == i]
            top1i, top5i = acc_i.mean(0).tolist()
            LOGGER.info(f"{c:>24}{acc_i.shape[0]:>12}{top1i:>12.3g}{top5i:>12.3g}")

        # Print results
        t = tuple(x.t / len(dataloader.dataset.samples) * 1e3 for x in dt)  # speeds per image
        shape = (1, 3, imgsz, imgsz)
        LOGGER.info(f"Speed: %.1fms pre-process, %.1fms inference, %.1fms post-process per image at shape {shape}" % t)
        LOGGER.info(f"Results saved to {colorstr('bold', save_dir)}")

    if not training:
        # Save predictions and true labels
        np.save(os.path.join(save_dir, "y_true.npy"), y_true)
        np.save(os.path.join(save_dir, "y_pred.npy"), y_pred)
    
        # Calculate metrics
        precision = precision_score(y_true, y_pred, average="weighted")
        recall = recall_score(y_true, y_pred, average="weighted")
        f1 = f1_score(y_true, y_pred, average="weighted")
        accuracy = accuracy_score(y_true, y_pred)
    
        # Print metrics
        LOGGER.info(f"Precision: {precision:.3f}, Recall: {recall:.3f}, F1-Score: {f1:.3f}, Accuracy: {accuracy:.3f}")
    
        # Detailed classification report
        LOGGER.info("\nClassification Report:\n" + classification_report(y_true, y_pred))

        return top1, top5, loss, accuracy

    return top1, top5, loss

def parse_opt():
    """Parses and returns command line arguments for YOLOv5 model evaluation and inference settings."""
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", type=str, default=ROOT / "../datasets/mnist", help="dataset path")
    parser.add_argument("--weights", nargs="+", type=str, default=ROOT / "yolov5s-cls.pt", help="model.pt path(s)")
    parser.add_argument("--batch-size", type=int, default=128, help="batch size")
    parser.add_argument("--imgsz", "--img", "--img-size", type=int, default=224, help="inference size (pixels)")
    parser.add_argument("--device", default="", help="cuda device, i.e. 0 or 0,1,2,3 or cpu")
    parser.add_argument("--workers", type=int, default=8, help="max dataloader workers (per RANK in DDP mode)")
    parser.add_argument("--verbose", nargs="?", const=True, default=True, help="verbose output")
    parser.add_argument("--project", default=ROOT / "runs/val-cls", help="save to project/name")
    parser.add_argument("--name", default="exp", help="save to project/name")
    parser.add_argument("--exist-ok", action="store_true", help="existing project/name ok, do not increment")
    parser.add_argument("--half", action="store_true", help="use FP16 half-precision inference")
    parser.add_argument("--dnn", action="store_true", help="use OpenCV DNN for ONNX inference")
    opt = parser.parse_args()
    print_args(vars(opt))
    return opt


def main(opt):
    """Executes the YOLOv5 model prediction workflow, handling argument parsing and requirement checks."""
    check_requirements(ROOT / "requirements.txt", exclude=("tensorboard", "thop"))
    run(**vars(opt))


if __name__ == "__main__":
    opt = parse_opt()
    main(opt)



Overwriting /kaggle/working/yolov5/classify/val.py


In [8]:
'''
for fold, (train_csv, test_csv) in enumerate(zip(TRAIN_CSV, TEST_CSV)):
    print(f"\n=== Starting Training for Fold {fold + 1} ===")
    
    # Organize dataset
    fold_dir = os.path.join(BASE_DIR, f"fold_{fold + 1}")
    train_dir, val_dir = organize_dataset(train_csv, IMAGES_DIR, fold_dir)
    
    # Create data.yaml
    data_yaml_path = os.path.join(fold_dir, "data.yaml")
    with open(data_yaml_path, "w") as f:
        f.write(f"""
train: {train_dir}
val: {val_dir}
nc: {len(os.listdir(train_dir))}
names: {os.listdir(train_dir)}
        """.strip())
    
    # Train YOLOv5
    print(f"Training YOLOv5 for Fold {fold + 1}...")
    train_command = [
        "python", "train.py",
        "--img", "640",
        "--batch", str(BATCH_SIZE),
        "--epochs", str(EPOCHS),
        "--data", data_yaml_path,
        "--weights", "yolov5s.pt",
        "--project", OUTPUT_DIR,
        "--name", f"fold_{fold + 1}",
        "--workers", "2",
        "--device", DEVICE
    ]
    try:
        process = subprocess.Popen(
            train_command,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )

        # Print the output line by line in real-time
        for line in process.stdout:
            print(line, end="")  # `end=""` avoids adding extra newlines

        # Wait for the process to finish
        process.wait()

        # Check for errors
        if process.returncode != 0:
            raise subprocess.CalledProcessError(process.returncode, train_command)
    except subprocess.CalledProcessError as e:
        print(f"Training failed for Fold {fold + 1}: {e}")
        continue

    # Validate YOLOv5
    print(f"Validating YOLOv5 for Fold {fold + 1}...")
    val_command = [
        "python", "classify/val.py",
        "--weights", os.path.join(OUTPUT_DIR, f"fold_{fold + 1}", "weights", "best.pt"),
        "--data", data_yaml_path,
        "--batch-size", str(BATCH_SIZE),
        "--imgsz", "640",
        "--project", OUTPUT_DIR,
        "--name", f"fold_{fold + 1}_val",
        "--device", DEVICE
    ]
    try:
        subprocess.run(val_command, check=True)
    except subprocess.CalledProcessError as e:
        print(f"Validation failed for Fold {fold + 1}: {e}")
        continue
    
    # Clean up the dataset for this fold
    print(f"Cleaning up dataset for Fold {fold + 1}...")
    shutil.rmtree(fold_dir, ignore_errors=True)

print("\n=== Cross-Validation Completed ===")
'''

'\nfor fold, (train_csv, test_csv) in enumerate(zip(TRAIN_CSV, TEST_CSV)):\n    print(f"\n=== Starting Training for Fold {fold + 1} ===")\n    \n    # Organize dataset\n    fold_dir = os.path.join(BASE_DIR, f"fold_{fold + 1}")\n    train_dir, val_dir = organize_dataset(train_csv, IMAGES_DIR, fold_dir)\n    \n    # Create data.yaml\n    data_yaml_path = os.path.join(fold_dir, "data.yaml")\n    with open(data_yaml_path, "w") as f:\n        f.write(f"""\ntrain: {train_dir}\nval: {val_dir}\nnc: {len(os.listdir(train_dir))}\nnames: {os.listdir(train_dir)}\n        """.strip())\n    \n    # Train YOLOv5\n    print(f"Training YOLOv5 for Fold {fold + 1}...")\n    train_command = [\n        "python", "train.py",\n        "--img", "640",\n        "--batch", str(BATCH_SIZE),\n        "--epochs", str(EPOCHS),\n        "--data", data_yaml_path,\n        "--weights", "yolov5s.pt",\n        "--project", OUTPUT_DIR,\n        "--name", f"fold_{fold + 1}",\n        "--workers", "2",\n        "--device",

In [9]:
import torch.multiprocessing as mp
mp.set_start_method("spawn", force=True)

In [10]:
def plot_training_history(results_csv_path):
    """
    Plots training and validation loss, and top-1 and top-5 accuracy from YOLOv5 classification training logs.

    Parameters:
    - results_csv_path (str): Path to the `results.csv` file containing training logs.

    Returns:
    - None: Displays the plots.
    """
    # Check if results.csv exists
    if not os.path.exists(results_csv_path):
        print(f"Results file not found at: {results_csv_path}")
        return

    # Load the results.csv file
    df = pd.read_csv(results_csv_path)

    # Plot training and validation loss
    plt.figure(figsize=(12, 6))
    plt.plot(df['epoch'], df['train_loss'], label='Train Loss', marker='o')
    plt.plot(df['epoch'], df['val_loss'], label='Validation Loss', marker='o')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid()
    plt.show()

    # Plot top-1 and top-5 accuracy
    plt.figure(figsize=(12, 6))
    plt.plot(df['epoch'], df['top1_acc'], label='Top-1 Accuracy', marker='o')
    plt.plot(df['epoch'], df['top5_acc'], label='Top-5 Accuracy', marker='o')
    plt.title('Top-1 and Top-5 Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid()
    plt.show()

In [11]:
from PIL import Image
import os

def find_mismatched_images(image_dir):
    mismatched_files = []
    for root, _, files in os.walk(image_dir):
        for file in files:
            file_path = os.path.join(root, file)
            try:
                with Image.open(file_path) as img:
                    if img.format != "JPEG" and file.lower().endswith(".jpg"):
                        mismatched_files.append(file_path)
            except Exception:
                mismatched_files.append(file_path)
    return mismatched_files

# Verify remaining files
def verify_images(image_dir):
    valid_count = 0
    invalid_count = 0
    for root, _, files in os.walk(image_dir):
        for file in files:
            file_path = os.path.join(root, file)
            try:
                with Image.open(file_path) as img:
                    # Check if the format is JPEG
                    if img.format == "JPEG":
                        valid_count += 1  # Count as valid
                    else:
                        os.remove(file_path)  # Remove mismatched file
            except Exception as e:
                print(f"Error opening file {file_path}: {e}")
                os.remove(file_path)  # Remove invalid file
                invalid_count += 1  # Count as invalid
    return valid_count, invalid_count


# Train fold 1

In [ ]:
fold_dir = os.path.join(BASE_DIR, "car_brand_dataset")
train_dir, val_dir = organize_dataset(TRAIN_CSV[0], TEST_CSV[0], IMAGES_DIR, fold_dir)

In [ ]:
# Example usage
image_dir = "/kaggle/working/car_brand_dataset"  # Update this path

# Find mismatched files
mismatched_files = find_mismatched_images(image_dir)
print(f"Number of mismatched files: {len(mismatched_files)}")

# Verify valid and invalid files
valid_count, invalid_count = verify_images(image_dir)
print(f"Number of valid files: {valid_count}")
print(f"Number of invalid files: {invalid_count}")

In [ ]:
!python -m torch.distributed.run \
  --nproc_per_node 2 \
  classify/train.py \
  --model yolov5s-cls.pt \
  --data /kaggle/working/car_brand_dataset \
  --epochs 20 \
  --batch-size 128 \
  --workers 2 \
  --device 0,1

In [20]:
!python classify/val.py \
  --weights runs/train-cls/exp4/weights/best.pt \
  --data /kaggle/working/car_brand_dataset \
  --batch-size 128

classify/val: data=/kaggle/working/car_brand_dataset, weights=['runs/train-cls/exp4/weights/best.pt'], batch_size=128, imgsz=224, device=, workers=8, verbose=True, project=runs/val-cls, name=exp, exist_ok=False, half=False, dnn=False
YOLOv5 🚀 v7.0-394-g86fd1ab2 Python-3.10.12 torch-2.4.1+cu121 CUDA:0 (Tesla T4, 15095MiB)

Fusing layers... 
Model summary: 117 layers, 4178217 parameters, 0 gradients, 10.4 GFLOPs
validating:   0%|          | 0/50 [00:00<?, ?it/s]/kaggle/working/yolov5/classify/val.py:116: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=device.type != "cpu"):
validating: 100%|██████████| 50/50 [00:39<00:00,  1.26it/s]
                   Class      Images    top1_acc    top5_acc
                     all        6399       0.887       0.988
                       0         835        0.84       0.992
                       1         491       0.849       0.992
   

In [ ]:
!rm -rf /kaggle/working/car_brand_dataset

# Train fold 2

In [13]:
fold_dir = os.path.join(BASE_DIR, "car_brand_dataset")
train_dir, val_dir = organize_dataset(TRAIN_CSV[1], TEST_CSV[1], IMAGES_DIR, fold_dir)

In [14]:
# Example usage
image_dir = "/kaggle/working/car_brand_dataset"  # Update this path

# Find mismatched files
mismatched_files = find_mismatched_images(image_dir)
print(f"Number of mismatched files: {len(mismatched_files)}")

# Verify valid and invalid files
valid_count, invalid_count = verify_images(image_dir)
print(f"Number of valid files: {valid_count}")
print(f"Number of invalid files: {invalid_count}")

Number of mismatched files: 3347
Number of valid files: 31944
Number of invalid files: 0


In [15]:
!python -m torch.distributed.run \
  --nproc_per_node 2 \
  classify/train.py \
  --model yolov5s-cls.pt \
  --data /kaggle/working/car_brand_dataset \
  --epochs 20 \
  --batch-size 128 \
  --workers 2 \
  --device 0,1

Error decoding JSON from /root/.config/Ultralytics/settings.json. Starting with an empty dictionary.
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
wandb: WARNING ⚠️ wandb is deprecated and will be removed in a future release. See supported integrations at https://github.com/ultralytics/yolov5#integrations.
2025-01-17 03:42:05.020576: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-01-17 03:42:05.020576: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-01

In [16]:
!python classify/val.py \
  --weights runs/train-cls/exp2/weights/best.pt \
  --data /kaggle/working/car_brand_dataset \
  --batch-size 128

classify/val: data=/kaggle/working/car_brand_dataset, weights=['runs/train-cls/exp2/weights/best.pt'], batch_size=128, imgsz=224, device=, workers=8, verbose=True, project=runs/val-cls, name=exp, exist_ok=False, half=False, dnn=False
YOLOv5 🚀 v7.0-397-gde62f93c Python-3.10.12 torch-2.4.1+cu121 CUDA:0 (Tesla T4, 15095MiB)

Fusing layers... 
Model summary: 117 layers, 4178217 parameters, 0 gradients, 10.4 GFLOPs
validating:   0%|          | 0/51 [00:00<?, ?it/s]/kaggle/working/yolov5/classify/val.py:116: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=device.type != "cpu"):
validating: 100%|██████████| 51/51 [00:40<00:00,  1.26it/s]
                   Class      Images    top1_acc    top5_acc
                     all        6417       0.881       0.984
                       0         832       0.839       0.986
                       1         491       0.839        0.98
   

In [17]:
!rm -rf /kaggle/working/car_brand_dataset

# Train fold 3

In [18]:
fold_dir = os.path.join(BASE_DIR, "car_brand_dataset")
train_dir, val_dir = organize_dataset(TRAIN_CSV[2], TEST_CSV[2], IMAGES_DIR, fold_dir)

In [19]:
# Example usage
image_dir = "/kaggle/working/car_brand_dataset"  # Update this path

# Find mismatched files
mismatched_files = find_mismatched_images(image_dir)
print(f"Number of mismatched files: {len(mismatched_files)}")

# Verify valid and invalid files
valid_count, invalid_count = verify_images(image_dir)
print(f"Number of valid files: {valid_count}")
print(f"Number of invalid files: {invalid_count}")

Number of mismatched files: 3347
Number of valid files: 31944
Number of invalid files: 0


In [20]:
!python -m torch.distributed.run \
  --nproc_per_node 2 \
  classify/train.py \
  --model yolov5s-cls.pt \
  --data /kaggle/working/car_brand_dataset \
  --epochs 20 \
  --batch-size 128 \
  --workers 2 \
  --device 0,1

wandb: WARNING ⚠️ wandb is deprecated and will be removed in a future release. See supported integrations at https://github.com/ultralytics/yolov5#integrations.
2025-01-17 04:56:53.485783: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-01-17 04:56:53.514459: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-01-17 04:56:53.527299: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-01-17 04:56:53.538585: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been register

In [21]:
!python classify/val.py \
  --weights runs/train-cls/exp4/weights/best.pt \
  --data /kaggle/working/car_brand_dataset \
  --batch-size 128

classify/val: data=/kaggle/working/car_brand_dataset, weights=['runs/train-cls/exp4/weights/best.pt'], batch_size=128, imgsz=224, device=, workers=8, verbose=True, project=runs/val-cls, name=exp, exist_ok=False, half=False, dnn=False
YOLOv5 🚀 v7.0-397-gde62f93c Python-3.10.12 torch-2.4.1+cu121 CUDA:0 (Tesla T4, 15095MiB)

Fusing layers... 
Model summary: 117 layers, 4178217 parameters, 0 gradients, 10.4 GFLOPs
validating:   0%|          | 0/50 [00:00<?, ?it/s]/kaggle/working/yolov5/classify/val.py:116: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=device.type != "cpu"):
validating: 100%|██████████| 50/50 [00:37<00:00,  1.33it/s]
                   Class      Images    top1_acc    top5_acc
                     all        6369       0.875       0.988
                       0         837        0.82       0.987
                       1         467       0.809       0.981
   

In [22]:
!rm -rf /kaggle/working/car_brand_dataset

# Train fold 4

In [23]:
fold_dir = os.path.join(BASE_DIR, "car_brand_dataset")
train_dir, val_dir = organize_dataset(TRAIN_CSV[3], TEST_CSV[3], IMAGES_DIR, fold_dir)

In [24]:
# Example usage
image_dir = "/kaggle/working/car_brand_dataset"  # Update this path

# Find mismatched files
mismatched_files = find_mismatched_images(image_dir)
print(f"Number of mismatched files: {len(mismatched_files)}")

# Verify valid and invalid files
valid_count, invalid_count = verify_images(image_dir)
print(f"Number of valid files: {valid_count}")
print(f"Number of invalid files: {invalid_count}")

Number of mismatched files: 3347
Number of valid files: 31944
Number of invalid files: 0


In [25]:
!python -m torch.distributed.run \
  --nproc_per_node 2 \
  classify/train.py \
  --model yolov5s-cls.pt \
  --data /kaggle/working/car_brand_dataset \
  --epochs 20 \
  --batch-size 128 \
  --workers 2 \
  --device 0,1

wandb: WARNING ⚠️ wandb is deprecated and will be removed in a future release. See supported integrations at https://github.com/ultralytics/yolov5#integrations.
2025-01-17 06:11:37.826137: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-01-17 06:11:37.826137: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-01-17 06:11:37.850274: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-01-17 06:11:37.850274: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been register

In [26]:
!python classify/val.py \
  --weights runs/train-cls/exp6/weights/best.pt \
  --data /kaggle/working/car_brand_dataset \
  --batch-size 128

classify/val: data=/kaggle/working/car_brand_dataset, weights=['runs/train-cls/exp6/weights/best.pt'], batch_size=128, imgsz=224, device=, workers=8, verbose=True, project=runs/val-cls, name=exp, exist_ok=False, half=False, dnn=False
YOLOv5 🚀 v7.0-397-gde62f93c Python-3.10.12 torch-2.4.1+cu121 CUDA:0 (Tesla T4, 15095MiB)

Fusing layers... 
Model summary: 117 layers, 4178217 parameters, 0 gradients, 10.4 GFLOPs
validating:   0%|          | 0/50 [00:00<?, ?it/s]/kaggle/working/yolov5/classify/val.py:116: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=device.type != "cpu"):
validating: 100%|██████████| 50/50 [00:38<00:00,  1.31it/s]
                   Class      Images    top1_acc    top5_acc
                     all        6368       0.877       0.986
                       0         819       0.817        0.99
                       1         464       0.834       0.972
   

In [27]:
!rm -rf /kaggle/working/car_brand_dataset

# Train fold 5

In [28]:
fold_dir = os.path.join(BASE_DIR, "car_brand_dataset")
train_dir, val_dir = organize_dataset(TRAIN_CSV[4], TEST_CSV[4], IMAGES_DIR, fold_dir)

In [29]:
# Example usage
image_dir = "/kaggle/working/car_brand_dataset"  # Update this path

# Find mismatched files
mismatched_files = find_mismatched_images(image_dir)
print(f"Number of mismatched files: {len(mismatched_files)}")

# Verify valid and invalid files
valid_count, invalid_count = verify_images(image_dir)
print(f"Number of valid files: {valid_count}")
print(f"Number of invalid files: {invalid_count}")

Number of mismatched files: 3347
Number of valid files: 31944
Number of invalid files: 0


In [30]:
!python -m torch.distributed.run \
  --nproc_per_node 2 \
  classify/train.py \
  --model yolov5s-cls.pt \
  --data /kaggle/working/car_brand_dataset \
  --epochs 20 \
  --batch-size 128 \
  --workers 2 \
  --device 0,1

wandb: WARNING ⚠️ wandb is deprecated and will be removed in a future release. See supported integrations at https://github.com/ultralytics/yolov5#integrations.
2025-01-17 07:39:37.147382: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-01-17 07:39:37.147388: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-01-17 07:39:37.171191: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-01-17 07:39:37.171191: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been register

In [31]:
!python classify/val.py \
  --weights runs/train-cls/exp8/weights/best.pt \
  --data /kaggle/working/car_brand_dataset \
  --batch-size 128

classify/val: data=/kaggle/working/car_brand_dataset, weights=['runs/train-cls/exp8/weights/best.pt'], batch_size=128, imgsz=224, device=, workers=8, verbose=True, project=runs/val-cls, name=exp, exist_ok=False, half=False, dnn=False
YOLOv5 🚀 v7.0-397-gde62f93c Python-3.10.12 torch-2.4.1+cu121 CUDA:0 (Tesla T4, 15095MiB)

Fusing layers... 
Model summary: 117 layers, 4178217 parameters, 0 gradients, 10.4 GFLOPs
validating:   0%|          | 0/50 [00:00<?, ?it/s]/kaggle/working/yolov5/classify/val.py:116: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=device.type != "cpu"):
validating: 100%|██████████| 50/50 [00:39<00:00,  1.28it/s]
                   Class      Images    top1_acc    top5_acc
                     all        6391        0.88       0.984
                       0         827       0.815       0.977
                       1         491       0.792       0.953
   

# Cross Validate Result

In [32]:
cross_val_metrics = [0.89, 0.88, 0.88, 0.88, 0.88]

In [33]:
mean_accuracy = sum(cross_val_metrics) / len(cross_val_metrics)
print("Mean Accuracy:", mean_accuracy)

Mean Accuracy: 0.882


In [34]:
import numpy as np

std_deviation = np.std(cross_val_metrics)
print("Standard Deviation:", std_deviation)

Standard Deviation: 0.0040000000000000036


# Test

In [32]:
from PIL import Image
import torch
from models.common import DetectMultiBackend
from utils.general import check_img_size
from utils.augmentations import letterbox
import numpy as np

from PIL import Image
import torch
from models.common import DetectMultiBackend
from utils.general import check_img_size
from utils.augmentations import letterbox
import numpy as np

def predict_image(image_path, weights, imgsz=224, device='cpu'):
    # Convert device to torch.device
    device = torch.device(device)

    # Load model
    model = DetectMultiBackend(weights, device=device)
    stride = model.stride
    imgsz = check_img_size(imgsz, s=stride)

    # Load and preprocess image
    img = Image.open(image_path).convert('RGB')
    img = letterbox(np.array(img), imgsz, stride=stride, auto=True)[0]
    img = np.transpose(img, (2, 0, 1))  # HWC to CHW
    img = np.ascontiguousarray(img).astype(np.float32) / 255.0  # Normalize to [0, 1] and cast to float32

    # Convert to tensor
    img_tensor = torch.from_numpy(img).unsqueeze(0).to(device)

    # Predict
    model.eval()
    with torch.no_grad():
        pred = model(img_tensor)
    class_index = torch.argmax(pred, dim=1).item()
    confidence = torch.softmax(pred, dim=1)[0, class_index].item()
    
    # Map index to class name
    class_names = model.names
    class_label = class_names[class_index]
    
    return class_label, confidence


image_path = "/kaggle/input/testdata/download.jpg"  # Replace with your image path
weights_path = "runs/train-cls/exp4/weights/best.pt"  # Replace with your trained weights path
device = 'cuda' if torch.cuda.is_available() else 'cpu'

predicted_label, confidence_score = predict_image(image_path, weights_path, device=device)

categories = {
    0: 'Others',
    1: 'Honda',
    2: 'Mazda',
    3: 'Mitsubishi',
    4: 'Suzuki',
    5: 'Toyota',
    6: 'Hyundai',
    7: 'KIA',
    8: 'VinFast'
}

print(f"Predicted Label: {categories.get(int(predicted_label))}")
print(f"Confidence Score: {confidence_score:.2f}")


Fusing layers... 
Model summary: 117 layers, 4178217 parameters, 0 gradients, 10.4 GFLOPs


Predicted Label: Toyota
Confidence Score: 0.98


In [ ]:
def read_yolo_train_history(csv_file_path):
    history = pd.read_csv(csv_file_path)
    
    return history

def plot_yolo_history(history, curSplit):
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(history['epoch'], history['train_loss'], label="Training Loss")
    plt.plot(history['epoch'], history['val_loss'], label="Validation Loss")
    plt.title(f'Loss Function Split {curSplit}')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    # Vẽ accuracy
    plt.subplot(1, 2, 2)
    plt.plot(history['epoch'], history['train_accuracy'], label="Training Accuracy")
    plt.plot(history['epoch'], history['val_accuracy'], label="Validation Accuracy")
    plt.title(f'Accuracy Split {curSplit}')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.show()